In [15]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

df = pd.read_csv("../data/processed/food_safety_clean (1).csv")
y = df["general_verdict"]
text = df["ingredients_clean"].str.lower()
labels = ["Safe", "Limit", "Avoid"]

cv = StratifiedKFold(5, shuffle=True, random_state=0)
tfidf_logistic = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True),
    LogisticRegression(max_iter=2000, class_weight="balanced"),
)

groups = df["brand_clean"]
pred_g = cross_val_predict(tfidf_logistic, text, y, cv=GroupKFold(5), groups=groups)
print(classification_report(y, pred_g, labels=labels))

df = pd.read_csv("../data/processed/food_safety_clean (1).csv")
y = df["general_verdict"]
text = df["ingredients_clean"].str.lower()

tfidf_logistic = Pipeline([
    ('tfidfvectorizer', TfidfVectorizer()),
    ('logisticregression', LogisticRegression(max_iter=1000))
])
# Train the model on ALL rows (this is for inspection only, not for scoring)
model = tfidf_logistic.fit(text, y)

vec = model.named_steps["tfidfvectorizer"]
clf = model.named_steps["logisticregression"]
words = vec.get_feature_names_out()

for i, cls in enumerate(clf.classes_):
    top = clf.coef_[i].argsort()[-15:][::-1]
    print(f"\nTop 15 words pushing toward {cls}:")
    print([words[j] for j in top])

# Predictions for every row, each made by a model that did NOT train on that row
pred = cross_val_predict(tfidf_logistic, text, y, cv=cv)

print(confusion_matrix(y, pred, labels=labels))
print("Avoid -> Safe rate:", round(((y == "Avoid") & (pred == "Safe")).sum() / (y == "Avoid").sum(), 4))

bad = df[(y == "Avoid") & (pred == "Safe")]
print("\nNumber of dangerous mistakes:", len(bad))

pd.set_option("display.max_colwidth", 120)
bad[["product_name", "ingredients_clean", "sugars_g", "salt_g", "reasons_general"]].head(5)

df["brand_clean"].value_counts().head(5)

cm = confusion_matrix(y, pred_g, labels=labels)
print(cm)
print("Avoid -> Safe:", ((y == "Avoid") & (pred_g == "Safe")).sum())
print("Avoid -> Safe or Limit:", ((y == "Avoid") & (pred_g != "Avoid")).sum())

              precision    recall  f1-score   support

        Safe       0.66      0.78      0.72       283
       Limit       0.64      0.59      0.61       540
       Avoid       0.79      0.79      0.79       703

    accuracy                           0.71      1526
   macro avg       0.70      0.72      0.71      1526
weighted avg       0.71      0.71      0.71      1526


Top 15 words pushing toward Avoid:
['sugar', 'fat', 'butter', 'ins', 'jaggery', 'and', 'hydrogenated', 'honey', 'refined', 'vegetable', 'glucose', 'ii', 'palm', 'salt', 'powder']

Top 15 words pushing toward Limit:
['sunflower', 'raisins', 'dates', 'oil', '282', 'sweeteners', 'nutmeg', 'gluten', 'pumpkin', 'anti', 'yeast', 'bran', 'seeds', 'rock', 'salt']

Top 15 words pushing toward Safe:
['culture', 'oats', 'rice', 'rolled', 'basmati', 'wheat', 'active', 'water', 'tea', 'turmeric', 'chilli', 'masoor', '224', 'durum', 'poppy']
[[168 104  11]
 [ 35 352 153]
 [  3 116 584]]
Avoid -> Safe rate: 0.0043

Number of 